# Stage 1 — Understanding the Levin & Yehudai dataset

**Purpose.** Establish exactly what the published dataset contains before anything is built on it.
No models, no engineered features, no decisions. Only: what are the columns, what do they hold, and
what does that allow.

**Why a whole notebook.** The dataset is the foundation of the three stages that follow —
reproducing their models, fine-tuning CodeBERT on the same labels, and applying the result to our
corpus. The record describes the file in prose, not by column name, so the schema has to be read
off the file itself.

**What the record claims** (10.5281/zenodo.835534):

- 1,151 commits manually labelled `c` corrective, `p` perfective, `a` adaptive, following
  Mockus & Votta (ICSM 2000)
- source-code changes per commit using Fluri & Gall's taxonomy (ICPC 2006) — **48 types**, already
  extracted, so ChangeDistiller does not need to be run
- a binary indication of whether each commit contains keywords from a pre-computed set — **20
  stems**, listed in the paper at §4 step 9
- 11 Java projects: RxJava, hbase, elasticsearch, intellij-community, hadoop, drools, kotlin,
  restlet-framework-java, orientdb, camel, spring-framework
- CSV separated by `#` rather than a comma, because commit messages contain commas

Every one of those is checked against the file below.

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 1. Load

Local files first, since Zenodo is not always reachable. Put `levin_commits.csv`, or an archive
containing it, in `WORK` (see the setup cell).

In [1]:
import csv, json, re, zipfile, urllib.request, urllib.error, collections
from pathlib import Path
import pandas as pd
import numpy as np

LEVIN_REC = "835534"                 # public record, no token needed
LEVIN = Path("levin"); LEVIN.mkdir(exist_ok=True)
LOOK_IN = [Path("."), LEVIN]          # WORK (the current directory) and WORK/levin
DELIM_SUFFIX = (".csv", ".tsv", ".txt")


def gather_local():
    """Copy any delimited file, or extract one from an archive, into levin/."""
    found = 0
    for d in LOOK_IN:
        if not d.exists():
            continue
        for p in sorted(d.iterdir()):
            if not p.is_file():
                continue
            if p.suffix.lower() in DELIM_SUFFIX and p.parent != LEVIN:
                (LEVIN / p.name).write_bytes(p.read_bytes()); found += 1
            elif p.suffix.lower() == ".zip":
                try:
                    with zipfile.ZipFile(p) as z:
                        for info in z.infolist():
                            n = Path(info.filename).name
                            if n.lower().endswith(DELIM_SUFFIX) and not (LEVIN / n).exists():
                                (LEVIN / n).write_bytes(z.read(info)); found += 1
                except zipfile.BadZipFile:
                    pass
    return found


def fetch_zenodo():
    try:
        with urllib.request.urlopen(
                f"https://zenodo.org/api/records/{LEVIN_REC}", timeout=60) as r:
            meta = json.load(r)
    except Exception as e:
        print(f"   Zenodo unreachable ({e})"); return
    for f in meta.get("files", []):
        name = f.get("key") or f.get("filename")
        dest = LEVIN / name
        if not dest.exists():
            print(f"   downloading {name} ...", end=" ", flush=True)
            try:
                urllib.request.urlretrieve(
                    f["links"].get("self") or f["links"].get("download"), dest)
                print("done")
            except Exception as e:
                print(f"FAILED: {e}")


have = [p for p in LEVIN.glob("*") if p.suffix.lower() in DELIM_SUFFIX]
if not have:
    n = gather_local()
    print(f"local: {n} file(s) collected" if n else "local: nothing found, trying Zenodo")
    if not n:
        fetch_zenodo()
    have = [p for p in LEVIN.glob("*") if p.suffix.lower() in DELIM_SUFFIX]

if not have:
    raise SystemExit("No delimited file found. Upload levin_commits.csv (or its archive).")

for p in have:
    print(f"   {p.name}  ({p.stat().st_size/1024:.0f} KB)")

local: nothing found, trying Zenodo
   downloading 1151-commits-labeled-with-maintenance-activities.csv ... done
   1151-commits-labeled-with-maintenance-activities.csv  (366 KB)


## 2. Parse

Two settings matter, and both were arrived at by the file disagreeing with a naive read.

**The separator is sniffed, not assumed.** A wrong separator collapses everything into one column,
so the one producing most columns is unambiguously right.

**Quoting is disabled.** Pandas treats `"` at the start of a field as opening a quoted region and
consumes across newlines until the closing quote, merging two physical lines into one record —
whose comment is then two commits' messages concatenated, with one commit lost entirely. A
field-count check cannot see this, because both lines are individually well formed. `QUOTE_NONE`
takes the file literally, which is what a `#`-separated export of commit messages needs.

`on_bad_lines` is left at its default so a malformed line raises rather than disappearing.

In [2]:
def sniff(path):
    """Try each separator; the right one yields many columns rather than one."""
    out = {}
    for sep in ["#", ",", "\t", ";", "|"]:
        try:
            out[sep] = pd.read_csv(path, sep=sep, engine="python", quoting=csv.QUOTE_NONE,
                                   on_bad_lines="skip", nrows=200).shape[1]
        except Exception:
            out[sep] = 0
    return out


path = max(have, key=lambda p: p.stat().st_size)
tried = sniff(path)
print(f"{path.name}   columns produced by each separator:")
for sep, n in sorted(tried.items(), key=lambda x: -x[1]):
    print(f"   {sep!r:<6}{n:>5}")

SEP = max(tried, key=tried.get)
print(f"\nusing {SEP!r} with quoting disabled\n")

D = pd.read_csv(path, sep=SEP, engine="python", quoting=csv.QUOTE_NONE)

print(f"{D.shape[0]:,} rows x {D.shape[1]} columns")
print(f"record claims 1,151 commits -> {'matches' if len(D) == 1151 else 'DIFFERS, see below'}")

1151-commits-labeled-with-maintenance-activities.csv   columns produced by each separator:
   '#'      72
   ','       1
   '\t'      1
   ';'       1
   '|'       1

using '#' with quoting disabled

1,151 rows x 72 columns
record claims 1,151 commits -> matches


## 3. Parse integrity

Three things a row count alone would not reveal.

In [3]:
raw_lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
n_fields = raw_lines[0].count(SEP) + 1

print(f"file lines (excluding header) : {len(raw_lines) - 1}")
print(f"rows parsed                   : {len(D)}")
print(f"fields in header              : {n_fields}\n")

bad = [(i, ln) for i, ln in enumerate(raw_lines[1:], start=2)
       if ln.strip() and ln.count(SEP) + 1 != n_fields]
blank = sum(1 for ln in raw_lines[1:] if not ln.strip())

if bad:
    print(f"{len(bad)} line(s) with an unexpected field count:\n")
    for i, ln in bad[:10]:
        got = ln.count(SEP) + 1
        print(f"   line {i}: {got} fields ({got - n_fields:+d})")
        print(f"      {ln[:150]}{'…' if len(ln) > 150 else ''}\n")
    if len(bad) > 10:
        print(f"   ... and {len(bad) - 10} more")
else:
    print("every non-blank line carries the expected field count")
if blank:
    print(f"blank lines: {blank}")

accounted = len(raw_lines) - 1 - blank
if accounted != len(D):
    print(f"\n{accounted - len(D)} line(s) did not become a row. Something is consuming lines --")
    print("inspect the lines listed above before going further.")

# A shifted row shows up as a malformed value in a column whose content is constrained.
def is_texty(s):
    """Object or string dtype -- pandas 3 reports `str` where earlier versions report `object`."""
    return pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s)

print("\n\nSIGNS OF COLUMN SHIFT\n")
checks = []
ID_GUESS = next((c for c in D.columns if is_texty(D[c])
                 and D[c].dropna().astype(str).str.fullmatch(r"[0-9a-f]{7,40}").mean() > 0.5), None)
LAB_GUESS = next((c for c in D.columns if is_texty(D[c])
                  and set(D[c].dropna().astype(str).str.strip().str.lower().unique()) <= {"c","p","a"}
                  and D[c].nunique() == 3), None)
if ID_GUESS:
    ok = D[ID_GUESS].astype(str).str.fullmatch(r"[0-9a-f]{7,40}")
    checks += [(f"values in {ID_GUESS!r} that are not hex hashes", int((~ok).sum())),
               (f"nulls in {ID_GUESS!r}", int(D[ID_GUESS].isna().sum()))]
if LAB_GUESS:
    ok = D[LAB_GUESS].astype(str).str.strip().str.lower().isin({"c","p","a"})
    checks += [(f"values in {LAB_GUESS!r} outside {{c,p,a}}", int((~ok).sum())),
               (f"nulls in {LAB_GUESS!r}", int(D[LAB_GUESS].isna().sum()))]
num = D.select_dtypes("number")
if len(num.columns):
    checks += [("rows where every numeric feature is 0", int((num.sum(axis=1) == 0).sum())),
               ("negative values in numeric features", int((num < 0).sum().sum())),
               ("nulls in numeric features", int(num.isna().sum().sum()))]

for name, n in checks:
    print(f"   {name:<48}{n:>6}   {'ok' if n == 0 else 'INVESTIGATE'}")
if checks and all(n == 0 for _, n in checks):
    print("\nNo shifted rows.")

file lines (excluding header) : 1151
rows parsed                   : 1151
fields in header              : 72

every non-blank line carries the expected field count


SIGNS OF COLUMN SHIFT

   values in 'commitId' that are not hex hashes         0   ok
   nulls in 'commitId'                                  0   ok
   values in 'label' outside {c,p,a}                    0   ok
   nulls in 'label'                                     0   ok
   rows where every numeric feature is 0                0   ok
   negative values in numeric features                  0   ok
   nulls in numeric features                            0   ok

No shifted rows.


## 4. The schema

Every column, its type, how many values it takes, and an example.

In [4]:
rows = []
for c in D.columns:
    s = D[c]
    sample = str(s.dropna().iloc[0]) if s.notna().any() else ""
    rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "distinct": s.nunique(dropna=True),
        "nulls": int(s.isna().sum()),
        "binary": bool(s.nunique(dropna=True) <= 2
                       and set(s.dropna().unique()) <= {0, 1, "0", "1"}),
        "example": sample[:48] + ("…" if len(sample) > 48 else ""),
    })
SCHEMA = pd.DataFrame(rows)
pd.set_option("display.max_rows", 200, "display.width", 200)
print(SCHEMA.to_string(index=False))

                          column  dtype  distinct  nulls  binary                                     example
                        commitId object      1149      0   False    03336a980d738426fb76202750befa48f3c1599e
                         project object        11      0   False                            ReactiveX-RxJava
                         comment object      1137      0   False Add marble diagrams for Single operators.--
                           label object         3      0   False                                           p
  ADDING_ATTRIBUTE_MODIFIABILITY  int64         4      0   False                                           0
       ADDING_CLASS_DERIVABILITY  int64         2      0    True                                           0
    ADDING_METHOD_OVERRIDABILITY  int64         6      0   False                                           0
                ADDITIONAL_CLASS  int64         8      0   False                                           0
        ADDITIONAL_

## 5. Labels

The record states `c` / `p` / `a`. Whichever column holds exactly those is the label.

In [5]:
LABEL_COL = next((c for c in D.columns
                  if set(D[c].dropna().astype(str).str.strip().str.lower().unique()) <= {"c","p","a"}
                  and D[c].nunique() == 3), None)
print(f"label column: {LABEL_COL}")

if LABEL_COL is None:
    print("\nNo column holds exactly {c, p, a}. Candidates with few distinct values:\n")
    print(SCHEMA[SCHEMA.distinct <= 5].to_string(index=False))
else:
    NAMES = {"c": "corrective", "p": "perfective", "a": "adaptive"}
    dist = D[LABEL_COL].astype(str).str.strip().str.lower().map(NAMES).value_counts()
    print(f"\n{dist.to_string()}\n")
    print(f"majority class: {dist.idxmax()} at {dist.max()/len(D)*100:.1f}%")
    print("The paper reports a No Information Rate of 43% (table 8), so this should land near")
    print("43% if the label column is being read correctly. A classifier predicting only the")
    print("majority class would score that accuracy while learning nothing, which is why the")
    print("paper reports kappa alongside accuracy.")

label column: label

label
corrective    500
perfective    404
adaptive      247

majority class: corrective at 43.4%
The paper reports a No Information Rate of 43% (table 8), so this should land near
43% if the label column is being read correctly. A classifier predicting only the
majority class would score that accuracy while learning nothing, which is why the
paper reports kappa alongside accuracy.


## 6. Commit identity

Stage 3a retrieves each commit's diff from its project repository, which needs a **hash** and a
**project**. Both are detected by content rather than by name.

In [6]:
PROJECTS = ["RxJava", "hbase", "elasticsearch", "intellij-community", "hadoop", "drools",
            "kotlin", "restlet-framework-java", "orientdb", "camel", "spring-framework"]

hash_cols, project_cols = [], []
for c in D.columns:
    if not is_texty(D[c]):
        continue
    s = D[c].dropna().astype(str)
    if s.empty:
        continue
    if s.str.fullmatch(r"[0-9a-f]{7,40}").mean() > 0.9:
        hash_cols.append(c)
    vals = set(s.str.lower().unique())
    if 1 < len(vals) <= 20 and any(p.lower() in v for p in PROJECTS for v in vals):
        project_cols.append(c)

print(f"commit-hash columns : {hash_cols or 'NONE FOUND'}")
print(f"project columns     : {project_cols or 'NONE FOUND'}")

if project_cols:
    vc = D[project_cols[0]].value_counts()
    print(f"\n{vc.to_string()}\n")
    print(f"{D[project_cols[0]].nunique()} projects (record names 11)")
    unprefixed = [p for p in PROJECTS if p not in set(vc.index)]
    if unprefixed:
        print(f"\nNames here do not all match the record's list -- {unprefixed} appear under a")
        print("different form. Stage 3 needs an explicit project -> clone URL map rather than")
        print("deriving one from this column.")
else:
    print("\nNo project column. Stage 3 would have to resolve each hash against all 11")
    print("repositories. Flagging now rather than in Stage 3.")

commit-hash columns : ['commitId']
project columns     : ['project']

project
drools                    114
hadoop                    112
hbase                     109
intellij-community        109
camel                     103
orientdb                  102
kotlin                    101
restlet-framework-java    101
ReactiveX-RxJava          100
elasticsearch             100
spring-framework          100

11 projects (record names 11)

Names here do not all match the record's list -- ['RxJava'] appear under a
different form. Stage 3 needs an explicit project -> clone URL map rather than
deriving one from this column.


## 7. Feature groups

Two sets are expected: **20 keyword indicators** and **48 change types**. The paper lists the
keyword stems explicitly (§4 step 9) as the ten most frequent words per activity — overlapping, so
20 distinct.

In [7]:
PAPER_KEYWORDS = {
    "corrective": ["fix", "test", "issu", "use", "fail", "bug", "report", "set", "error", "npe"],
    "perfective": ["test", "remov", "use", "fix", "refactor", "method", "chang", "add",
                   "improv", "new"],
    "adaptive":   ["support", "add", "implement", "new", "allow", "use", "method", "test",
                   "set", "chang"],
}
KEYWORD_UNION = sorted({w for v in PAPER_KEYWORDS.values() for w in v})
print(f"the paper's keyword stems: {len(KEYWORD_UNION)} distinct (paper states tuples of size 20)")
print("   " + ", ".join(KEYWORD_UNION))

norm = {c.lower().strip(): c for c in D.columns}
matched = {w: norm[w] for w in KEYWORD_UNION if w in norm}
print(f"\nmatched as columns by exact name: {len(matched)}/{len(KEYWORD_UNION)}")
missing_kw = [w for w in KEYWORD_UNION if w not in matched]
if missing_kw:
    print(f"   not found: {', '.join(missing_kw)}")

the paper's keyword stems: 20 distinct (paper states tuples of size 20)
   add, allow, bug, chang, error, fail, fix, implement, improv, issu, method, new, npe, refactor, remov, report, set, support, test, use

matched as columns by exact name: 20/20


In [8]:
binary_cols = SCHEMA.loc[SCHEMA.binary, "column"].tolist()
numeric_cols = [c for c in D.columns
                if pd.api.types.is_numeric_dtype(D[c]) and c not in binary_cols]
text_cols = [c for c in D.columns
             if is_texty(D[c])
             and c not in hash_cols
             and c not in project_cols
             and int(SCHEMA.loc[SCHEMA.column == c, "distinct"].iloc[0]) > 50]

print(f"binary columns  : {len(binary_cols)}")
print(f"other numeric   : {len(numeric_cols)}")
print(f"free-text       : {len(text_cols)}   {text_cols}")

kw_cols = sorted(matched.values())
change_cols = [c for c in (binary_cols + numeric_cols) if c not in kw_cols]
print(f"\ncandidate change-type columns: {len(change_cols)}   (record implies 48)\n")
for c in change_cols[:60]:
    print(f"   {c:<44}{D[c].min()}-{D[c].max()}")
if len(change_cols) > 60:
    print(f"   ... and {len(change_cols)-60} more")

# The paper is explicit that the two groups are encoded differently: keyword features are capped
# at one per commit, while every occurrence of a change type is counted. If that does not hold,
# the grouping is wrong and Stage 2 would train on a mis-split feature space.
print("\n\nENCODING CHECK\n")
kw_max = int(D[kw_cols].max().max()) if kw_cols else 0
ch_max = int(D[change_cols].max().max()) if change_cols else 0
print(f"   largest value in a keyword column      {kw_max:>6}   "
      f"{'ok, capped at 1 as the paper states' if kw_max == 1 else 'EXPECTED 1'}")
print(f"   largest value in a change-type column  {ch_max:>6}   "
      f"{'ok, counted in full' if ch_max > 1 else 'EXPECTED > 1'}")

binary columns  : 22
other numeric   : 46
free-text       : 1   ['comment']

candidate change-type columns: 48   (record implies 48)

   ADDING_CLASS_DERIVABILITY                   0-1
   PARENT_INTERFACE_CHANGE                     0-0
   ADDING_ATTRIBUTE_MODIFIABILITY              0-5
   ADDING_METHOD_OVERRIDABILITY                0-24
   ADDITIONAL_CLASS                            0-12
   ADDITIONAL_FUNCTIONALITY                    0-315
   ADDITIONAL_OBJECT_STATE                     0-57
   ALTERNATIVE_PART_DELETE                     0-16
   ALTERNATIVE_PART_INSERT                     0-37
   ATTRIBUTE_RENAMING                          0-240
   ATTRIBUTE_TYPE_CHANGE                       0-55
   CLASS_RENAMING                              0-2
   COMMENT_DELETE                              0-69
   COMMENT_INSERT                              0-29
   COMMENT_MOVE                                0-12
   COMMENT_UPDATE                              0-150
   CONDITION_EXPRESSION_CHANGE     

## 8. Data quality

Three checks that would each change how Stage 2 is built.

In [9]:
print("DUPLICATE COMMITS\n")
if hash_cols:
    dup = int(D[hash_cols[0]].duplicated().sum())
    print(f"   duplicate values in {hash_cols[0]!r}: {dup}")
    if dup:
        print("   The same commit in both a train and a test split leaks. Stage 2 should group")
        print("   by commit id when splitting, or drop the repeats -- either is fine, but it")
        print("   needs deciding rather than ignoring.")
        print(D[D[hash_cols[0]].duplicated(keep=False)]
              [[hash_cols[0]] + project_cols + ([LABEL_COL] if LABEL_COL else [])]
              .sort_values(hash_cols[0]).to_string(index=False))
else:
    print(f"   duplicate rows (all columns): {int(D.duplicated().sum())}")

print("\n\nMISSING VALUES\n")
miss = SCHEMA[SCHEMA.nulls > 0][["column", "nulls"]]
print(miss.to_string(index=False) if len(miss) else "   none")

print("\n\nCOMMITS WITH NO KEYWORD AT ALL\n")
if kw_cols:
    none_kw = int((D[kw_cols].fillna(0).astype(float).sum(axis=1) == 0).sum())
    print(f"   {none_kw} of {len(D)} ({none_kw/len(D)*100:.1f}%)")
    print("   The paper reports 28% to 34.8% in this position. These commits have an all-zero")
    print("   keyword vector, so a message-only model has no signal for them and falls back to")
    print("   the majority class. This is why their compound model uses source-code changes as")
    print("   the second half, and why Stage 4 substitutes the diff for that half.")

if text_cols:
    col = text_cols[0]
    print(f"\n\nIS {col!r} RAW OR PREPROCESSED?\n")
    for ch, why in [(SEP, "the separator -- its absence would mean the export stripped it"),
                    ('"', "quote characters"),
                    ("\n", "embedded newlines")]:
        n = int(D[col].astype(str).str.contains(re.escape(ch), regex=True).sum())
        print(f"   comments containing {ch!r:<6}{n:>6}   ({why})")
    print("\n   Commit messages routinely reference issues as '#123'. If none contain the")
    print("   separator, the export stripped it, and Stage 4 would train on cleaned text while")
    print("   applying to raw messages -- a distribution mismatch worth stating.")

DUPLICATE COMMITS

   duplicate values in 'commitId': 2
   The same commit in both a train and a test split leaks. Stage 2 should group
   by commit id when splitting, or drop the repeats -- either is fine, but it
   needs deciding rather than ignoring.
                                commitId          project label
65a5e2cc46b2c04591aa59b7a85751479a7dbd0b           hadoop     a
65a5e2cc46b2c04591aa59b7a85751479a7dbd0b           hadoop     a
f26b7b2321ba561c947da92510e95e22f059351c ReactiveX-RxJava     c
f26b7b2321ba561c947da92510e95e22f059351c ReactiveX-RxJava     p


MISSING VALUES

   none


COMMITS WITH NO KEYWORD AT ALL

   323 of 1151 (28.1%)
   The paper reports 28% to 34.8% in this position. These commits have an all-zero
   keyword vector, so a message-only model has no signal for them and falls back to
   the majority class. This is why their compound model uses source-code changes as
   the second half, and why Stage 4 substitutes the diff for that half.


IS 'comment' RAW O

## 9. What this enables

Written from what was observed, not from the record's description.

In [10]:
print("=" * 72)
print("STAGE 1 SUMMARY")
print("=" * 72)
print(f"rows                      {len(D):,}  (record: 1,151)")
print(f"columns                   {D.shape[1]}")
print(f"separator                 {SEP!r}, quoting disabled")
print(f"label column              {LABEL_COL}")
print(f"keyword columns           {len(kw_cols)}  (paper: 20)")
print(f"change-type columns       {len(change_cols)}  (record: 48)")
print(f"commit hash               {hash_cols[0] if hash_cols else 'NOT PRESENT'}")
print(f"project                   {project_cols[0] if project_cols else 'NOT PRESENT'}")
print(f"commit message            {text_cols[0] if text_cols else 'NOT PRESENT'}")
print()
print("Stage 2 can reproduce their models from these columns alone -- ChangeDistiller does not")
print("need to be run, because the change types are already extracted here.")
print()
print("Stage 3 needs a hash and a project per commit to retrieve diffs. Anything reported as")
print("NOT PRESENT above is a gap to solve before Stage 3 starts.")

D.to_pickle("levin_dataset.pkl")
SCHEMA.to_csv("levin_schema.csv", index=False)
json.dump({"sep": SEP, "label": LABEL_COL,
           "keywords": kw_cols, "changes": change_cols,
           "hash": hash_cols[0] if hash_cols else None,
           "project": project_cols[0] if project_cols else None,
           "message": text_cols[0] if text_cols else None},
          open("levin_columns.json", "w"), indent=1)
print("\n   levin_dataset.pkl, levin_schema.csv and levin_columns.json written for Stage 2")

STAGE 1 SUMMARY
rows                      1,151  (record: 1,151)
columns                   72
separator                 '#', quoting disabled
label column              label
keyword columns           20  (paper: 20)
change-type columns       48  (record: 48)
commit hash               commitId
project                   project
commit message            comment

Stage 2 can reproduce their models from these columns alone -- ChangeDistiller does not
need to be run, because the change types are already extracted here.

Stage 3 needs a hash and a project per commit to retrieve diffs. Anything reported as
NOT PRESENT above is a gap to solve before Stage 3 starts.

   levin_dataset.pkl, levin_schema.csv and levin_columns.json written for Stage 2
